# Magnitude Estimation

This notebook demonstrates the **`swe.magnitude`** subpackage for computing earthquake magnitudes using three methods:

1. **Local magnitude (ML)** via Wood-Anderson simulation (`swe.magnitude.local_magnitude`)
2. **Duration magnitude (Md)** from coda duration (`swe.magnitude.duration_magnitude`)
3. **Relative moments** from spectral ratios (`swe.magnitude.relative_moments_from_streams`)

**Dataset**: Synthetic waveforms with known amplitudes to verify magnitude computation.
For real data, swap in FDSN-downloaded waveforms with instrument responses.

In [ ]:
from __future__ import annotations

import logging

import matplotlib.pyplot as plt
import numpy as np
from obspy import Stream, Trace, UTCDateTime

import swe.magnitude
import swe.viz

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")

%matplotlib inline

## Configuration

In [ ]:
# Synthetic parameters
SAMPLING_RATE = 100.0
DURATION = 30.0  # seconds
NPTS = int(SAMPLING_RATE * DURATION)

STATIONS = ["STA1", "STA2", "STA3", "STA4"]
DISTANCES_KM = [10.0, 25.0, 50.0, 80.0]  # hypocentral distances
AMPLITUDES = [500.0, 200.0, 50.0, 20.0]  # peak amplitude scales

## 1. Generate Synthetic Waveforms

We create synthetic traces with known peak amplitudes that decay with distance,
mimicking a real earthquake's amplitude-distance relationship.

In [ ]:
rng = np.random.default_rng(42)
st = Stream()

for station, amp in zip(STATIONS, AMPLITUDES):
    # Synthetic seismogram: tapered sine burst + noise
    t = np.arange(NPTS) / SAMPLING_RATE
    signal = amp * np.sin(2 * np.pi * 5.0 * t) * np.exp(-0.3 * t)
    noise = rng.normal(0, amp * 0.05, NPTS)
    data = (signal + noise).astype(np.float64)

    tr = Trace(
        data=data,
        header={
            "network": "XX",
            "station": station,
            "channel": "BHZ",
            "sampling_rate": SAMPLING_RATE,
            "starttime": UTCDateTime(2024, 1, 1),
        },
    )
    st.append(tr)

print(st)
fig = swe.viz.plot_stream(st)
fig.set_size_inches(12, 6)
plt.tight_layout()
plt.show()

## 2. Wood-Anderson Amplitude

The `wood_anderson_amplitude` function simulates the classic Wood-Anderson
seismograph response and returns the peak displacement amplitude in mm.

In [ ]:
for tr in st:
    wa_amp = swe.magnitude.wood_anderson_amplitude(tr)
    print(f"{tr.stats.station}: WA amplitude = {wa_amp:.4f} mm")

## 3. Local Magnitude (ML)

Compute ML using the Hutton & Boore (1987) attenuation relation:

$$ML = \log_{10}(A) + a \cdot \log_{10}(r/100) + b \cdot (r - 100) + c + 3.0$$

In [ ]:
result = swe.magnitude.local_magnitude(st, DISTANCES_KM)

print(f"Mean ML: {result.ml:.2f}")
print(f"\nPer-station ML:")
for name, ml_i, res in zip(result.station_names, result.ml_per_station, result.residuals):
    print(f"  {name}: ML = {ml_i:.2f}  (residual = {res:+.3f})")

# Plot ML vs distance
fig, ax = plt.subplots(figsize=(8, 5))
valid = ~np.isnan(result.ml_per_station)
ax.scatter(np.array(DISTANCES_KM)[valid], result.ml_per_station[valid], s=80, zorder=5)
ax.axhline(result.ml, color="r", ls="--", label=f"Mean ML = {result.ml:.2f}")
ax.set_xlabel("Hypocentral Distance (km)")
ax.set_ylabel("ML")
ax.set_title("Local Magnitude vs Distance")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Duration Magnitude (Md)

Duration magnitude uses the coda duration (time from the P-wave onset until
the signal amplitude drops below a threshold) instead of peak amplitude.

In [ ]:
# Measure coda durations
for tr in st:
    coda = swe.magnitude.coda_duration(tr, threshold_factor=3.0)
    print(f"{tr.stats.station}: coda duration = {coda:.2f} s")

# Compute duration magnitude
md_result = swe.magnitude.duration_magnitude(
    st, distances_km=DISTANCES_KM, threshold_factor=3.0,
)
print(f"\nMean Md: {md_result.md:.2f}")
for name, md_i in zip(md_result.station_names, md_result.md_per_station):
    print(f"  {name}: Md = {md_i:.2f}")

## 5. Relative Moments

Relative moments compare spectral amplitudes between events to estimate
moment ratios - useful when absolute calibration is unavailable.

In [ ]:
# Create a second "event" with different amplitudes
st2 = Stream()
for station, amp in zip(STATIONS, [a * 0.5 for a in AMPLITUDES]):
    t = np.arange(NPTS) / SAMPLING_RATE
    signal = amp * np.sin(2 * np.pi * 5.0 * t) * np.exp(-0.3 * t)
    data = (signal + rng.normal(0, amp * 0.05, NPTS)).astype(np.float64)
    tr = Trace(
        data=data,
        header={
            "network": "XX", "station": station, "channel": "BHZ",
            "sampling_rate": SAMPLING_RATE,
            "starttime": UTCDateTime(2024, 1, 1, 0, 5),
        },
    )
    st2.append(tr)

rel_result = swe.magnitude.relative_moments_from_streams(
    [st, st2], freqmin=1.0, freqmax=20.0,
)
print(f"Relative moments: {rel_result.moments}")
print(f"Reference event index: {rel_result.reference_index}")

## Summary

| Method | Function | Input | Output |
|--------|----------|-------|--------|
| Local ML | `local_magnitude()` | WA amplitudes + distances | Mean ML + per-station |
| Duration Md | `duration_magnitude()` | Coda durations + distances | Mean Md + per-station |
| Relative | `relative_moments_from_streams()` | Multi-event streams | Moment ratios |

For real data, attach instrument responses to traces via `trace.stats.response`
before calling `wood_anderson_amplitude()` for proper deconvolution.